# Step 05: Merge publication and journal information

Input:

    - Gender and Affiliation
        - data/interim/publication/3_first_gender_affiliation.csv
        - data/interim/publication/3_last_gender_affiliation.csv
        
    - Publication
        data/interim/publication/1_AllPublications.csv

    - Editor
        data/raw/journal/1_EditorGender.csv
        data/raw/journal/2_schimagojr.csv

        
Output:
    - data/interim/publication/5_first_author_publication_info.csv
    - data/interim/publication/5_last_author_publication_info.csv


Purpose:
    - Generate disciplinary domians and subfields indicators
    - Merge Publication information (citation, team size,...)
    - Merge Journal information (Editor Gender and SJR rankings, H index...)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# -----------------------------
# Paths
# -----------------------------


PROJECT_ROOT = Path(__file__).resolve().parents[2]

ALL_PUB_FILE = PROJECT_ROOT / "data" / "interim" / "publication"/ "1_AllPublications.csv"

FIRST_GENDER_PATH = PROJECT_ROOT / "data" / "interim" / "publication"/ "3_first_gender_affiliation.csv"
LAST_GENDER_PATH = PROJECT_ROOT / "data" / "interim" / "publication"/ "3_last_gender_affiliation.csv"


EDITOR_GENDER_PATH = PROJECT_ROOT / "data" / "raw" / "journal" / "1_EditorGender.csv"
SJR_PATH = PROJECT_ROOT / "data" / "raw" / "journal" / "2_scimagojr.csv"


FIRST_OUT = PROJECT_ROOT / "data" / "interim" / "publication"/ "5_first_author_publication_info.csv"
LAST_OUT = PROJECT_ROOT / "data" / "interim" / "publication"/ "5_last_author_publication_info.csv"

# 1 Publication Information

In [ ]:
use_col = ['index_num','Source Title', 'Publication Year',
           'Author Full Names', 'Times Cited, All Databases', 
           'WoS Categories'
          
          ]
paper = pd.read_csv(ALL_PUB_FILE, usecols=use_col)

unique_journals = paper['Source Title'].nunique()
print("Number of unique journal:", unique_journals)

In [ ]:
# 1.1 Calculate the total number of authors
# count the number of author by using Full Author Name
paper['total_author_num'] = paper['Author Full Names'].str.split(';').str.len()
paper

In [ ]:
# 1.2 Disciplinary subfield indicators though WoS categories
# field indicator


paper['woscat'] = paper['WoS Categories'].str.replace(" ", "")

# Initialize the field column
paper['field'] = None

# Agricultural Sciences
paper.loc[paper['woscat'].str.contains("Agriculture,Dairy&AnimalScience|Agriculture,Multidisciplinary|Agronomy|Horticulture", na=False), 'field'] = 1

# Arts & Humanities, Interdisciplinary
paper.loc[paper['woscat'].str.contains("AreaStudies|AsianStudies|CulturalStudies|History&PhilosophyOfScience|Humanities,Multidisciplinary|Medieval&RenaissanceStudies", na=False), 'field'] = 2

# Biology & Biochemistry
paper.loc[paper['woscat'].str.contains("Anatomy&Morphology|Mycology|BiochemicalResearchMethods|Biochemistry&MolecularBiology|Biology|Biophysics|Biotechnology&AppliedMicrobiology|Cell&TissueEngineering|CellBiology|DevelopmentalSciences|Entomology|EvolutionaryBiology|Fisheries|FoodScience&Technology|Cenetics&Heredity|Marine&FreshwaterBiology|Mathematical&ComputationalBiology|Microbiology|mycology|Neurosciences|Parasitology|Pathology|Pharmacology&Pharmacy|Physiology|ReproductiveBiology|Toxicology|Virology|Zoology", na=False), 'field'] = 3

# Chemistry
paper.loc[paper['woscat'].str.contains("BiochemicalResearchMethod|Chemistry,Analytical|Chemistry,Applied|Chemistry,Inorganic&Nuclear|Chemistry,Medicinal|Chemistry,Organic|Chemistry,Physical|Electrochemistry|Energy&Fuels|Geochemistry&Geophysics|Geology|Mineralogy|Oceanography|PolymerScience|Spectroscopy", na=False), 'field'] = 4

# Clinical Medicine
paper.loc[paper['woscat'].str.contains("Allergy|Andrology|Anesthesiology|Audiology&Speech-languagePathology|BehavioralScience|Cardiac&CardiovascularSystems|ClinicalNeurology|CriticalCareMedicine|Dentistry,OralSurgery&Medicine|Dermatology|EmergencyMedicine|Endocrinology&Metabolism|Engineering,Biological|Gastroenterology&Hepatology|Genetics&Heredity|Geriatrics&Gerontology|HealthCareSciences&Service|HealthPolicy&Service|Hematology|Immunology|InfectiousDiseases|Integrative&ComplementaryMedicine|MaterialsScience,Biomatrials|MedicalEthics|MedicalInformatics|MedicalLaboratoryTechnology", na=False), 'field'] = 5
paper.loc[paper['woscat'].str.contains("Medicine,General&Internal|PeripheralVascularDisease|Medicine,Legal|Medicine,Research&Experimental|Neuroimaging|Nursing|Nutrition&Dietetics|Obstetrics&Gynecology|Oncology|Ophthalmology|Orthopedics|Otorhinolaryngology|Pediatrics|PeripheralVascularDiseases|PrimaryHealthCare|Public,Environmental&OccupationalHealth|Radiology,NuclearMedicine&MedicalImaging|Rehabilitation|RespiratorySystem|Rheumatology|SportScience|SubstanceAbuse|Surgery|Transplantation|TropicalMedicine|Urology&Nephrology", na=False), 'field'] = 5

# Computer Science
paper.loc[paper['woscat'].str.contains("Automation&ControlSystems|ComputerScience,ArtificialIntelligence|ComputerScience,Cybernetics|ComputerScience,Hardware&Architecture|ComputerScience,InformationSystems|ComputerScience,InterdisciplinaryApplications|ComputerScience,SoftwareEngineering|ComputerScience,Theory&Methods|InformationScience&LibraryScience|Robotics", na=False), 'field'] = 6

# Economics & Business
paper.loc[paper['woscat'].str.contains("AgriculturalEconomics&Policy|Business|Business,Finance|DevelopmentStudies|Economics|HealthCareScience&Services|IndustrialRelations&Labor|Management|OperationsResearch&ManagementScience", na=False), 'field'] = 7

# Engineering
paper.loc[paper['woscat'].str.contains("AgriculturalEngineering|Construction&BuildingTechnology|Engineering,Aerospace|Engineering,Biomedical|Engineering,Chemical|Engineering,Civil|Engineering,Geological|Engineering,Industrial|Engineering,Manufacturing|Engineering,Marine|Engineering,Mechanical|Engineering,Multidisciplinary|Engineering,Ocean|Engineering,Petroleum|ImagingScience&PhotographicTechnology|Instruments&Instrumentation|Mechanics|Metallurgy&MetallurgicalEngineering|Microscopy|Mining&MineralProcessing|NuclearScience&Technology|Optics|Telecommunications|TransportationScience&Technology", na=False), 'field'] = 8

# Environment/Ecology
paper.loc[paper['woscat'].str.contains("BiodiversityConservation|Ecology|Engineering,Environmental|Forestry|Green&SustainableScience&Technology", na=False), 'field'] = 9

# Geosciences
paper.loc[paper['woscat'].str.contains("Geography,Physical|Geosciences,Multidisciplinary|Meteorology&AtmosphericSciences|Paleontology|RemoteSensing|SoilScience|WaterResources", na=False), 'field'] = 10

# History & Archaeology
paper.loc[paper['woscat'].str.contains("Archaeology|History", na=False), 'field'] = 11

# Literature & Language
paper.loc[paper['woscat'].str.contains("Classics|Folklore|Language&Linguistics|Linguistics|LiteraryReviews|LiteraryTheory&Criticism|Literature|Literature,African,Australian,Canadian|Literature,American|Literature,BritishIsles|Literature,German,Dutch,Scandinavian|Literature,Romance|Literature,Slavic|Poetry", na=False), 'field'] = 12

# Materials Science
paper.loc[paper['woscat'].str.contains("Crystallography|Engineering,Electrical&Electronic|MaterialsScience,Biomaterials|MaterialsScience,Ceramics|MaterialsScience,Characterization&Testing|MaterialsScience,Coatings&Films|MaterialsScience,Composites|MaterialsScience,Multidisciplinary|MaterialsScience,Paper&Wood|MaterialsScience,Textiles|Nanoscience&Nanotechnology", na=False), 'field'] = 13

# Mathematics
paper.loc[paper['woscat'].str.contains("Logic|Mathematics|Mathematics,Applied|Mathematics,InterdisciplinaryApplications|Statistics&Probability", na=False), 'field'] = 14

# Multidisciplinary
paper.loc[paper['woscat'].str.contains("Chemistry,Multidisciplinary|EnvironmentalSciences|EnvironmentalStudies|Ergonomics|FamilyStudies|HistoryOfSocialSciences|MultidisciplinarySciences|SocialIssues", na=False), 'field'] = 15

# Philosophy & Religion
paper.loc[paper['woscat'].str.contains("Ethics|Philosophy|Religion|Women'sStudies", na=False), 'field'] = 16

# Physics
paper.loc[paper['woscat'].str.contains("Acoustics|Astronomy&Astrophysics|Physics,Applied|Physics,Atomic,Molecular&Chemical|Physics,CondensedMatter|Physics,Fluids&Plasmas|Physics,Mathematical|Physics,Multidisciplinary|Physics,Nuclear|Physics,Particles&Fields|QuantumScience&Technology|Thermodynamics", na=False), 'field'] = 17

# Plant & Animal Science
paper.loc[paper['woscat'].str.contains("Limnology|Ornithology|PlantSciences|VeterinarySciences", na=False), 'field'] = 18

# Psychiatry/Psychology
paper.loc[paper['woscat'].str.contains("BehavioralSciences|Gerontology|Psychiatry|Psychology|Psychology,Applied|Psychology,Biological|Psychology,Clinical|Psychology,Developmental|Psychology,Educational|Psychology,Experimental|Psychology,Mathematical|Psychology,Multidisciplinary|Psychology,Psychoanalysis|Psychology,Social", na=False), 'field'] = 19

# Social Sciences, General
paper.loc[paper['woscat'].str.contains("Anthropology|Communication|Criminology&Penology|Demography|Education&EducationalResearch|Education,ScientificDisciplines|Education,Special|EthnicStudies|Geography|HealthCareSciences&Services|HealthPolicy&Services|Hospitality,Leisure,Sport&Tourism|InternationalRelations|Law|PoliticalScience|PublicAdministration|Regional&UrbanPlanning|SocialSciences,Biomedical|SocialSciences,Interdisciplinary|SocialSciences,MathematicalMethods|SocialWork|Sociology|SportSciences|Transportation|UrbanStudies|WomensStudies", na=False), 'field'] = 20

# Visual & Performing Arts
paper.loc[paper['woscat'].str.contains("Architecture|Art|Dance|Film,Radio,Television|Music|Theater", na=False), 'field'] = 21

# Filter rows where 'field' is NaN, then get unique values from 'woscat'
unique_woscat = paper.loc[paper['field'].isna(), 'woscat'].unique()

# Print the result
print(unique_woscat)

num_missing = paper['field'].isna().sum()
print("Number of rows with NaN in field:", num_missing)

In [ ]:
# Generate 5 major disciplinary domains indicator
# Initialize the wos five category column
paper['category'] = None

#Arts & Humanities
paper.loc[paper['woscat'].str.contains("Architecture|Art|Arts&HumanitiesOtherTopics|AsianStudies|Classics|Dance|Film,Radio&Television|History|History&PhilosophyofScience|Literature|Music|Philosophy|Religion|Theater",na=False),'category'] = 1

#Life Sciences & Biomedicine
paper.loc[paper['woscat'].str.contains("Agriculture|Allergy|Anatomy&Morphology|Anesthesiology|Anthropology|Audiology&Speech-LanguagePathology|BehavioralSciences|Biochemistry&MolecularBiology|Biodiversity&Conservation|Biophysics|Biotechnology&AppliedMicrobiology|CardiovascularSystem&Cardiology|CellBiology|CriticalCareMedicine|Dentistry,OralSurgery&Medicine|Dermatology|DevelopmentalBiology|EmergencyMedicine",na=False),'category'] = 2
paper.loc[paper['woscat'].str.contains("Endocrinology&Metabolism|Entomology|EnvironmentalSciences&Ecology|EvolutionaryBiology|Fisheries|FoodScience&Technology|Forestry|Gastroenterology&Hepatology|General&InternalMedicine|Genetics&Heredity|Geriatrics&Gerontology|HealthCareSciences&Services|Hematology|Immunology|InfectiousDiseases|Integrative&ComplementaryMedicine|LegalMedicine|LifeSciencesBiomedicineOtherTopics|Marine&FreshwaterBiology|Mathematical&ComputationalBiology|MedicalEthics",na=False),'category'] = 2
paper.loc[paper['woscat'].str.contains("MedicalInformatics|MedicalLaboratoryTechnology|Microbiology|Mycology|Neurosciences&Neurology|Nursing|Nutrition&Dietetics|Obstetrics&Gynecology|Oncology|Ophthalmology|Orthopedics|Otorhinolaryngology|Paleontology|Parasitology|Pathology|Pediatrics|Pharmacology&Pharmacy|Physiology|PlantSciences|Psychiatry|Public,Environmental&OccupationalHealth|Radiology,NuclearMedicine&MedicalImaging|Rehabilitation|ReproductiveBiology|Research&ExperimentalMedicine|RespiratorySystem",na=False),'category'] = 2
paper.loc[paper['woscat'].str.contains("Rheumatology|SportSciences|SubstanceAbuse|Surgery|Toxicology|Transplantation|TropicalMedicine|Urology&Nephrology|VeterinarySciences|Virology|Zoology",na=False),'category'] = 2

#Physical Sciences
paper.loc[paper['woscat'].str.contains("Astronomy&Astrophysics|Chemistry|Crystallography|Electrochemistry|Geochemistry&Geophysics|Geology|Mathematics|Meteorology&AtmosphericSciences|Mineralogy|Mining&MineralProcessing|Oceanography|Optics|PhysicalGeography|Physics|PolymerScience|Thermodynamics|WaterResources",na=False),'category'] = 3



#Social Sciences
paper.loc[paper['woscat'].str.contains("Archaeology|AreaStudies|BiomedicalSocialSciences|Business&Economics|Communication|Criminology&Penology|CulturalStudies|Demography|DevelopmentStudies|Education&EducationalResearch|EthnicStudies|FamilyStudies|Geography|Government&Law|InternationalRelations|Linguistics|MathematicalMethodsInSocialSciences|Psychology|PublicAdministration|SocialIssues|SocialSciencesOtherTopics|SocialWork|Sociology|UrbanStudies|Women'sStudies",na=False),'category'] = 4


#Technology
paper.loc[paper['woscat'].str.contains("Acoustics|Automation&ControlSystems|ComputerScience|Construction&BuildingTechnology|Energy&Fuels|Engineering|ImagingScience&PhotographicTechnology|InformationScience&LibraryScience|Instruments&Instrumentation|MaterialsScience|Mechanics|Metallurgy&MetallurgicalEngineering|Microscopy|NuclearScience&Technology|OperationsResearch&ManagementScience|RemoteSensing|Robotics|Science&TechnologyOtherTopics|Spectroscopy|Telecommunications|Transportation",na=False),'category'] = 5


# Generate wos_id from woscat

paper['wos_id'], unique_labels = pd.factorize(paper['woscat'])


# Replace missing values in 'category' based on 'field'
paper.loc[paper['category'].isna() & paper['field'].isin([1, 3, 4, 5, 9, 18]), 'category'] = 2
paper.loc[paper['category'].isna() & paper['field'].isin([7, 19, 20]), 'category'] = 4
paper.loc[paper['category'].isna() & paper['field'].isin([10, 14]), 'category'] = 3
paper.loc[paper['category'].isna() & paper['field'].isin([16]), 'category'] = 1

# Filter rows where 'field' is NaN, then get unique values from 'woscat'
unique_field = paper.loc[paper['category'].isna(), 'field'].unique()

# Print the result
print(unique_field)


In [ ]:
# generate STEM indicator
paper['stem'] = np.where(paper['category'].isin([2, 3, 5]), 1, 0)

In [ ]:
paper = paper.drop(['WoS Categories','woscat'], axis=1)

In [ ]:
# 1.3 Summarize the publications in different domains and subfields
paper = paper.drop(['Author Full Names'], axis=1)
paper = paper.rename(columns={"Times Cited, All Databases": "pap_cited"})

# Count number of papers per year per field
paper['fld_sumwork'] = paper.groupby(['Publication Year', 'field'])['field'].transform('size')

# Calculate average citations per Publication Year and field
paper['fld_avgcited'] = paper.groupby(['Publication Year', 'field'])['pap_cited'].transform('mean')

In [ ]:
# Merge with first authorship
# Path to your CSV file

first = pd.read_csv(FIRST_GENDER_PATH)
print(first.columns)

first_merged = first.merge(paper, on='index_num', how='left')


In [ ]:
# Merge with last authorship
# Path to your CSV file

last = pd.read_csv(LAST_GENDER_PATH)
print(last.columns)
last

last_merged = last.merge(paper, on='index_num', how='left')
last_merged = last_merged.drop(['address'], axis=1)

# 2 Journal Information

In [ ]:

# 2.1 Editor Gender

use_col = ['Journal name', 'Publication Year','editor_Female_Ratio', 'editor_Has_Female']

editor1 = pd.read_excel(EDITOR_GENDER_PATH, usecols=use_col)
editor1 = editor1.rename(columns={'Journal name':'Source Title'})
editor1

# Ensure uniqueness in editor
editor_unique = editor1.drop_duplicates(subset=['Source Title', 'Publication Year'])

# sanity check: should be zero
print(editor_unique.duplicated(subset=['Source Title','Publication Year']).sum())

editor_unique = editor_unique.rename(columns={'editor_Has_Female':'has_female_editor','editor_Female_Ratio':'female_editor_ratio'})

In [ ]:
# 2.2 Merge with first authorship
first_editor = first_merged.merge(
    editor_unique,
    on=['Source Title','Publication Year'],
    how='left'
)

first_editor

In [ ]:
# 2.3 Merge with last authorship
last_editor = last_merged.merge(
    editor_unique,
    on=['Source Title','Publication Year'],
    how='left'
)

last_editor

In [ ]:
# 2.4 Other journal information
# Path to your CSV file
index = pd.read_csv(SJR_PATH)


index = index.rename(columns={'Sourceid':'jrn_id','Title':'Source Title','SJR':'jrn_sjr','H index':'jrn_hindex','Total Refs.':'jrn_totref','Ref. / Doc.':'jrn_refperdoc','%Female':'jrn_female_ratio','year':'Publication Year'})
index["Source Title"] = index["Source Title"].str.upper()


# Ensure uniqueness in editor
index = index.drop_duplicates(subset=['Source Title', 'Publication Year'])


# Merge with first authorship
first_index = first_editor.merge(
    index,
    on=['Source Title','Publication Year'],
    how='left'
)

first_index

# Merge with last authorship
last_index = last_editor.merge(
    index,
    on=['Source Title','Publication Year'],
    how='left'
)

last_index



# Fix the "," problen in records

cols = ["jrn_sjr", "jrn_refperdoc", "jrn_female_ratio"]

# Replace ',' with '.' and convert to float
for col in cols:
    first_index[col] = (
        first_index[col]
        .astype(str)            # make sure they're strings
        .str.replace(",", ".", regex=False)
        .astype(float)          # convert to float
    )
first_index

# Replace ',' with '.' and convert to float
for col in cols:
    last_index[col] = (
        last_index[col]
        .astype(str)            # make sure they're strings
        .str.replace(",", ".", regex=False)
        .astype(float)          # convert to float
    )
last_index

In [ ]:
first_index.to_csv(FIRST_OUT, index=False)
last_index.to_csv(LAST_OUT, index=False)